# Demo 02 : Interrupts in multi-agent workflows

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · in-a-subgraph

We compile a child workflow without its own saver and give the parent a checkpointer. The child's approval appears in the parent's result. We resume the parent with the same thread id.

In [2]:
import json
import operator
import sqlite3
import time
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Usage(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls += 1

    def on_llm_end(self, response, **kwargs):
        for batch in response.generations:
            for generation in batch:
                usage = generation.message.usage_metadata or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)

usage = Usage()
llm = make_llm(callbacks=[usage], max_retries=0, max_tokens=650)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
    conn.row_factory = sqlite3.Row
    work_order = dict(conn.execute("SELECT * FROM work_orders WHERE wo_id = ?", ("BW-7001",)).fetchone())
    part = dict(conn.execute("SELECT * FROM parts WHERE part_no = ?", ("CE-CF-02",)).fetchone())
print("work order:", work_order)
print("part:", part)

def config(label):
    return {"configurable": {"thread_id": f"07-{label}-{uuid4()}"}, "recursion_limit": 30}

work order: {'wo_id': 'BW-7001', 'fleet_no': 'CE12-2021-0117', 'opened': '2026-09-14', 'status': 'open', 'error_code': 'F-CE-110', 'summary': 'Traction system over-temperature during afternoon route', 'mechanic_id': 'M01'}
part: {'part_no': 'CE-CF-02', 'model': 'CE-12', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}


In [3]:
class Job(TypedDict):
    work_order: str
    draft: str
    approved: bool
    result: str

def draft_job(state: Job):
    return {"draft": llm.invoke(f"Write a short inspection proposal for Veenhaven Bus. Use only these facts: {state['work_order']}").content}

def approve_job(state: Job):
    answer = interrupt({"draft": state["draft"], "work_order": work_order["wo_id"]})
    return {"approved": answer["approved"]}

child_builder = StateGraph(Job)
child_builder.add_node("draft", draft_job)
child_builder.add_node("approval", approve_job)
child_builder.add_edge(START, "draft")
child_builder.add_edge("draft", "approval")
child_builder.add_edge("approval", END)
child = child_builder.compile()
parent_builder = StateGraph(Job)
parent_builder.add_node("specialist", child)
parent_builder.add_node("record", lambda s: {"result": s["draft"] if s["approved"] else "declined"})
parent_builder.add_edge(START, "specialist")
parent_builder.add_edge("specialist", "record")
parent_builder.add_edge("record", END)
parent = parent_builder.compile(checkpointer=InMemorySaver())
cfg = config("subgraph")
out = parent.invoke({"work_order": json.dumps(work_order), "draft": "", "approved": False, "result": ""}, cfg, version="v2")
print("parent payload:", out.interrupts[0].value)
print("parent next:", parent.get_state(cfg).next)
for task in parent.get_state(cfg, subgraphs=True).tasks:
    print("child next:", task.state.next)
assert out.interrupts and not out.value["result"]
out = parent.invoke(Command(resume={"approved": True}), cfg, version="v2")
print("parent result:", out.value["result"])
assert out.value["approved"] and out.value["result"]

parent payload: {'draft': '**Inspection Proposal — Veenhaven Bus**\n\nInspect fleet vehicle **CE12-2021-0117** under work order **BW-7001** for error code **F-CE-110**, reported as traction system over-temperature during an afternoon route. The work order was opened on **14 September 2026** and remains open. Assigned mechanic: **M01**.', 'work_order': 'BW-7001'}
parent next: ('specialist',)
child next: ('approval',)
parent result: **Inspection Proposal — Veenhaven Bus**

Inspect fleet vehicle **CE12-2021-0117** under work order **BW-7001** for error code **F-CE-110**, reported as traction system over-temperature during an afternoon route. The work order was opened on **14 September 2026** and remains open. Assigned mechanic: **M01**.


## b · in-a-subagent

We expose the child as a tool to a model supervisor. The nested interrupt passes through the tool node to the top-level caller. Resuming the supervisor re-enters the tool and delivers the reply to the child's interrupt.

In [4]:
tool_entries = []

@tool
def inspection_specialist(work_order_id: str) -> str:
    """Ask the inspection specialist to propose and approve work for a work order."""
    if work_order_id != work_order["wo_id"]:
        raise ValueError("Unknown work order")
    tool_entries.append(work_order_id)
    print("tool entered:", work_order_id)
    result = child.invoke({"work_order": json.dumps(work_order), "draft": "", "approved": False, "result": ""})
    return json.dumps({"approved": result["approved"], "draft": result["draft"]})

supervisor = create_agent(llm, tools=[inspection_specialist],
    system_prompt="Call inspection_specialist once for the requested work order. Report the returned approval and proposal. Do not call it again after a result.",
    checkpointer=InMemorySaver())
supervisor_cfg = config("subagent")
out = supervisor.invoke({"messages": [("human", "Request the inspection specialist's proposal for BW-7001.")]}, supervisor_cfg, version="v2")
print("top-level payload:", [item.value for item in out.interrupts])
assert len(out.interrupts) == 1
print("top-level next:", supervisor.get_state(supervisor_cfg).next)
print("tool entries before resume:", len(tool_entries))
out = supervisor.invoke(Command(resume={"approved": True}), supervisor_cfg, version="v2")
print("tool entries after resume:", len(tool_entries))
for message in out.value["messages"]:
    if isinstance(message, ToolMessage):
        print("specialist result:", message.content)
        assert json.loads(message.content)["approved"] is True
print("supervisor:", out.value["messages"][-1].content)
assert not out.interrupts and len(tool_entries) == 2

tool entered: BW-7001


top-level payload: [{'draft': '## Inspection Proposal — Veenhaven Bus\n\n**Work order:** BW-7001  \n**Fleet number:** CE12-2021-0117  \n**Opened:** 2026-09-14  \n**Status:** Open  \n**Error code:** F-CE-110  \n**Assigned mechanic:** M01  \n\nInspect the traction system following the reported over-temperature during an afternoon route. Record the findings and recommend any required next steps.', 'work_order': 'BW-7001'}]
top-level next: ('tools',)
tool entries before resume: 1
tool entered: BW-7001


tool entries after resume: 2
specialist result: {"approved": true, "draft": "## Inspection Proposal \u2014 Veenhaven Bus\n\n**Work order:** BW-7001  \n**Fleet number:** CE12-2021-0117  \n**Opened:** 2026-09-14  \n**Status:** Open  \n**Error code:** F-CE-110  \n**Assigned mechanic:** M01  \n\nInspect the traction system following the reported over-temperature during an afternoon route. Record the findings and recommend any required next steps."}
supervisor: **Approved:** Yes

**Proposal — BW-7001 (Veenhaven Bus):** Inspect the traction system following the reported over-temperature during an afternoon route. Record the findings and recommend any required next steps.


We leave nested invocation configuration to LangGraph's propagation. We do not create a separate child thread or turn its interrupt into a normal tool result. Tool code before the nested call runs again, so we put external actions after approval and use idempotency keys.

In [5]:
print("metrics:", json.dumps({"model": MODEL, "calls": usage.calls,
      "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
      "seconds": round(time.perf_counter() - started, 1)}))

metrics: {"model": "openai/gpt-6-luna", "calls": 4, "input_tokens": 537, "output_tokens": 245, "seconds": 11.5}


## Things to try

Optional : we can resume the top-level supervisor with approval set to false and inspect the tool result.